# Train LoRA — Llama 3.2 3B (Campaign Workflow AI)

## 1. Cài đặt & kiểm tra GPU

In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.46" "peft>=0.13" "trl>=0.12" bitsandbytes accelerate datasets huggingface_hub

## 2. Đăng nhập Hugging Face

**Cách tạo token (khuyến nghị):**

- Vào https://huggingface.co/settings/tokens → **Create new token**
- Chọn **Classic** (không phải Fine-grained) → quyền **Read**
- Hoặc Fine-grained: bật **"Access public gated repositories"**

In [ ]:
from huggingface_hub import login, HfApi, hf_hub_download
from getpass import getpass
import os

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

# Colab Secrets / session cũ có thể set HF_TOKEN sai → xóa trước khi login
for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
    os.environ.pop(key, None)

HF_TOKEN = getpass("HF token Classic Read (hf_...): ").strip()
if not HF_TOKEN:
    raise ValueError("Cần token. Tạo Classic Read: https://huggingface.co/settings/tokens")

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

api = HfApi()
try:
    info = api.model_info(BASE_MODEL, token=HF_TOKEN)
    config_path = hf_hub_download(BASE_MODEL, "config.json", token=HF_TOKEN)
    print(f"OK — {info.id}")
    print(f"Config: {config_path}")
except Exception as e:
    err = str(e)
    if "403" in err or "gated" in err.lower():
        raise RuntimeError(
            "403 — Token không đủ quyền tải Llama (gated repo).\n\n"
            "Làm lần lượt:\n"
            "1. https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct → Agree (cùng account)\n"
            "2. https://huggingface.co/settings/tokens → Create **Classic** token, quyền **Read**\n"
            "   (Fine-grained phải bật: Access public gated repositories)\n"
            "3. Colab: Runtime → Restart session → chạy lại cell này với token MỚI\n"
            "4. Nếu dùng Colab Secrets: xóa HF_TOKEN cũ hoặc cập nhật token mới\n"
            f"\nChi tiết: {err}"
        ) from e
    raise

## 3. Upload dataset


In [ ]:
from google.colab import files
import shutil
import zipfile
from pathlib import Path

DATASET_DIR = Path("/content/dataset")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "workflow-structure.train.jsonl",
    "workflow-structure.val.jsonl",
    "template-content.train.jsonl",
    "template-content.val.jsonl",
]

print("Chọn ml-dataset.zip hoặc từng file .jsonl...")
uploaded = files.upload()

for name, data in uploaded.items():
    if name.endswith(".zip"):
        zip_path = Path("/content/_upload.zip")
        zip_path.write_bytes(data)
        extract_to = Path("/content/_zip_extract")
        shutil.rmtree(extract_to, ignore_errors=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_to)
        for src in extract_to.rglob("*.jsonl"):
            shutil.copy2(src, DATASET_DIR / src.name)
        zip_path.unlink(missing_ok=True)
    elif name.endswith(".jsonl"):
        (DATASET_DIR / name).write_bytes(data)

print("\nDataset:")
for f in sorted(DATASET_DIR.glob("*.jsonl")):
    lines = sum(1 for _ in open(f, encoding="utf-8"))
    print(f"  {f.name:40s} {lines:4d} lines")

missing = [n for n in REQUIRED_FILES if not (DATASET_DIR / n).exists()]
if missing:
    raise FileNotFoundError(
        "Thiếu: " + ", ".join(missing) + "\n"
        "Trên máy: cd ml/dataset && zip ~/ml-dataset.zip *.jsonl"
    )

## 4. Hàm train LoRA (dùng chung)


In [ ]:
import json
import torch
from pathlib import Path
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

LORA_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def train_lora(
    *,
    task_name: str,
    train_file: str,
    val_file: str,
    output_dir: str,
    epochs: int = 3,
    batch_size: int = 2,
    max_length: int = 2048,
    learning_rate: float = 2e-4,
    lora_r: int = 16,
    lora_alpha: int = 32,
) -> Path:
    """Train QLoRA adapter; returns output path."""
    train_path = DATASET_DIR / train_file
    val_path = DATASET_DIR / val_file
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    hf_token = os.environ.get("HF_TOKEN")

    print(f"\n{'='*60}")
    print(f"Task: {task_name}")
    print(f"Base: {BASE_MODEL} (from Hugging Face)")
    print(f"Train: {train_path.name} | Val: {val_path.name}")
    print(f"Output: {out}")
    print(f"{'='*60}\n")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def format_row(row: dict) -> dict:
        text = tokenizer.apply_chat_template(
            row["messages"], tokenize=False, add_generation_prompt=False
        )
        return {"text": text}

    train_ds = Dataset.from_list([format_row(r) for r in load_jsonl(train_path)])
    val_ds = Dataset.from_list([format_row(r) for r in load_jsonl(val_path)])
    print(f"Samples — train: {len(train_ds)}, val: {len(val_ds)}")

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        token=hf_token,
    )

    lora = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=LORA_TARGETS,
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=str(out),
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=2,
            learning_rate=learning_rate,
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="epoch",
            max_length=max_length,
            dataset_text_field="text",
            bf16=torch.cuda.is_available(),
            report_to="none",
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(str(out))
    tokenizer.save_pretrained(str(out))

    # Lưu metadata để inference local biết base model
    (out / "base_model.txt").write_text(BASE_MODEL, encoding="utf-8")
    print(f"\nDone — saved to {out}")
    return out

print("train_lora() ready.")

## 5. Train workflow-structure LoRA

In [ ]:
WORKFLOW_OUT = train_lora(
    task_name="workflow-structure",
    train_file="workflow-structure.train.jsonl",
    val_file="workflow-structure.val.jsonl",
    output_dir="/content/adapters/workflow-lora",
    epochs=3,
    batch_size=2,
    max_length=2048,
)

## 6. Train template-content LoRA

Sinh subject + body email/SMS. Có thể giảm `epochs` xuống 2 nếu Colab sắp timeout.

In [ ]:
TEMPLATE_OUT = train_lora(
    task_name="template-content",
    train_file="template-content.train.jsonl",
    val_file="template-content.val.jsonl",
    output_dir="/content/adapters/template-lora",
    epochs=3,
    batch_size=4,
    max_length=1024,
)

## 7. Smoke test — thử generate workflow JSON

In [ ]:
import re
from peft import PeftModel

hf_token = os.environ.get("HF_TOKEN")
test_prompt = "Welcome new users: email immediately, SMS after 1 day"
user_content = (
    "Channels: email,sms,rcs,voice,voice_sms\n"
    "Categories: onboarding,abandoned_basket,reactivation\n\n"
    f"Prompt: {test_prompt}"
)
messages = [
    {"role": "system", "content": "You output only valid JSON for a marketing workflow. No markdown."},
    {"role": "user", "content": user_content},
]

tok = AutoTokenizer.from_pretrained(str(WORKFLOW_OUT), token=hf_token)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.bfloat16, token=hf_token,
)
model = PeftModel.from_pretrained(base, str(WORKFLOW_OUT))
model.eval()

text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=800, do_sample=True, temperature=0.7, pad_token_id=tok.eos_token_id)
decoded = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("--- Model output ---")
print(decoded[:2000])

match = re.search(r"\{[\s\S]*\}", decoded)
if match:
    try:
        obj = json.loads(match.group())
        print("\n--- JSON parse OK ---")
        print("name:", obj.get("name"))
        print("steps:", len(obj.get("steps", [])))
    except json.JSONDecodeError as e:
        print("\nJSON parse failed:", e)
else:
    print("\nNo JSON object found in output — có thể cần train thêm epoch hoặc thêm data.")

## 8. Tải adapter về máy

Giải nén vào project:
```
ml/adapters/workflow-lora/
ml/adapters/template-lora/
```

Sau đó trên máy: `huggingface-cli login` → `pnpm ml:serve` → `ML_USE_MOCK=false` → `pnpm dev`

In [ ]:
import shutil
from google.colab import files

for name, folder in [("workflow-lora", WORKFLOW_OUT), ("template-lora", TEMPLATE_OUT)]:
    archive = f"/content/{name}"
    shutil.make_archive(archive, "zip", str(folder))
    print(f"Downloading {name}.zip ...")
    files.download(f"{archive}.zip")

print("\nTrên máy local:")
print("  unzip workflow-lora.zip -d '.../ml/adapters/workflow-lora'")
print("  unzip template-lora.zip -d '.../ml/adapters/template-lora'")